# Text Redaction Workflow

Safe local wrapper for the `text-redact` CLI. This notebook does not implement detection or redaction logic; it only builds commands for the CLI workflows in this repository.

All `RUN_*` flags default to `False` so opening or running the notebook top-to-bottom does not process local videos, download models, or write run artifacts unless explicitly enabled.

In [ ]:
from pathlib import Path
import shlex
import subprocess

REPO_ROOT = Path.cwd()
INPUT_VIDEO = REPO_ROOT / "input-videos" / "sample.mp4"
OUTPUT_ROOT = REPO_ROOT / "runs" / "redactions"
RUN_ID = "notebook-local-test"

MODEL_NAME = "PP-OCRv5_server_det"
DEVICE = None
MASK_COLOR = "black"
EXPAND_PIXELS = 0.0
EXPAND_RATIO = 0.0

RUN_TEXT_REDACT_HELP = False
RUN_REDACT_VIDEO_HELP = False
RUN_REDACT_VIDEO = False

## Command Helpers

Commands are represented as argument lists so they can be inspected safely before execution.

In [ ]:
def format_command(command: list[str]) -> str:
    """Return a shell-readable command string for display."""
    return " ".join(shlex.quote(part) for part in command)


def run_command(command: list[str]) -> subprocess.CompletedProcess[str]:
    """Run one CLI command from the repository root."""
    return subprocess.run(command, cwd=REPO_ROOT, check=True, text=True)

## CLI Help

These cells are useful smoke checks for the installed CLI surface.

In [ ]:
text_redact_help_command = ["uv", "run", "text-redact", "--help"]
redact_video_help_command = ["uv", "run", "text-redact", "redact-video", "--help"]

print(format_command(text_redact_help_command))
print(format_command(redact_video_help_command))

if RUN_TEXT_REDACT_HELP:
    run_command(text_redact_help_command)

if RUN_REDACT_VIDEO_HELP:
    run_command(redact_video_help_command)

## Video Redaction

This command processes every frame offline and writes ignored local artifacts under `runs/redactions/<run-id>/`. PaddleOCR inference may download model weights and requires the project to be installed with the `paddleocr` extra.

In [ ]:
redact_video_command = [
    "uv",
    "run",
    "text-redact",
    "redact-video",
    str(INPUT_VIDEO),
    "--output-root",
    str(OUTPUT_ROOT),
    "--run-id",
    RUN_ID,
    "--mask-color",
    MASK_COLOR,
    "--expand-pixels",
    str(EXPAND_PIXELS),
    "--expand-ratio",
    str(EXPAND_RATIO),
    "--model-name",
    MODEL_NAME,
]

if DEVICE:
    redact_video_command.extend(["--device", DEVICE])

print(format_command(redact_video_command))

if RUN_REDACT_VIDEO:
    run_command(redact_video_command)

## Expected Artifacts

When `RUN_REDACT_VIDEO` is enabled, the workflow writes `predictions/text_regions.jsonl`, `redacted/<video-name>.mp4`, `summary.json`, and `summary.csv` under the configured run directory. These paths are ignored by git.